# core

> Verify step state management helpers for routes

In [ ]:
#| default_exp routes.core

In [ ]:
#| export
from typing import Dict, Any, Optional, NamedTuple

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore
from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_transcript_verify.models import VerifyStepState

# Type alias for state store
WorkflowStateStore = SQLiteWorkflowStateStore

# Debug flag for state tracing
DEBUG_VERIFY_STATE = False

## Verify Context

NamedTuple containing commonly-needed verify state values.

In [ ]:
#| export
class VerifyContext(NamedTuple):
    """Common verify state values loaded by handlers."""
    document_id: Optional[str]  # UUID of committed Document node
    media_path: Optional[str]  # Source media path for display context

## State Getters

In [ ]:
#| export
def _get_verify_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str  # Session identifier string
) -> VerifyStepState:  # Verify step state dictionary
    """Get the verify step state from the workflow state store."""
    workflow_state = state_store.get_state(workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    return step_states.get("verify", {})

def _get_review_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str  # Session identifier string
) -> Dict[str, Any]:  # Review step state dictionary
    """Get the review step state (for document_id fallback)."""
    workflow_state = state_store.get_state(workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    return step_states.get("review", {})

## Load Context

Loads verify state in a single call, with fallback to review state for document_id.

In [ ]:
#| export
def _load_verify_context(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str  # Session identifier string
) -> VerifyContext:  # Common verify state values
    """Load commonly-needed verify state values with review fallback."""
    verify_state = _get_verify_state(state_store, workflow_id, session_id)
    review_state = _get_review_state(state_store, workflow_id, session_id)
    
    # Try verify state first, fall back to review state
    document_id = verify_state.get("document_id") or review_state.get("document_id")
    media_path = verify_state.get("media_path") or review_state.get("media_path")
    
    if DEBUG_VERIFY_STATE:
        print(f"[VERIFY_STATE] document_id: {document_id}")
        print(f"[VERIFY_STATE] media_path: {media_path}")
    
    return VerifyContext(
        document_id=document_id,
        media_path=media_path,
    )

## State Updates

In [ ]:
#| export
def _update_verify_state(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    session_id:str,  # Session identifier string
    document_id:str=None,  # Document ID (None = don't change)
    media_path:str=None,  # Media path (None = don't change)
) -> None:
    """Update the verify step state in the workflow state store."""
    workflow_state = state_store.get_state(workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    verify_state = step_states.get("verify", {})
    
    # Update only provided fields
    if document_id is not None:
        verify_state["document_id"] = document_id
    if media_path is not None:
        verify_state["media_path"] = media_path
    
    step_states["verify"] = verify_state
    workflow_state["step_states"] = step_states
    
    if DEBUG_VERIFY_STATE:
        print(f"[VERIFY_STATE] Updated: {verify_state}")
    
    state_store.update_state(workflow_id, session_id, workflow_state)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()